# 月データ探索ツール

3つの公開データセット（クレーターの形・クレーターの推定年代・月面の温度）を、
**自分でX軸・Y軸を選びながら**散布図で見比べる教材です。

使い方：
1. 上から順にセルを実行する（Colabなら「ランタイム」→「すべてのセルを実行」）
2. 一番下に出てくるプルダウンで、データセット・X軸・Y軸・色分けを自由に選ぶ
3. まずは何も予想せず、いろいろな組み合わせを試してみる
4. 「気になる関係」が見つかったら、ワークシート（docs/worksheet.pdf）に書き出す

> 迷ったら、いちばん下の「問いのヒント」を参考にしてください。

In [ ]:
# ライブラリの読み込みと実行環境の確認
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and not os.path.exists('data'):
    print("\u26a0\ufe0f dataフォルダが見つかりません。")
    print("Colabでこのノートブックだけを開いた場合、データファイルは一緒に来ません。")
    print("次の2行のコメントを外して実行し、リポジトリごと取得してください：")
    print("  # !git clone https://github.com/<ユーザー名>/<リポジトリ名>.git")
    print("  # %cd <リポジトリ名>/notebooks")

In [ ]:
# 3つのデータセットを読み込む
craters = pd.read_csv('../data/craters_subset.csv') if os.path.exists('../data/craters_subset.csv') else pd.read_csv('data/craters_subset.csv')
deepcraters = pd.read_csv('../data/deepcraters.csv') if os.path.exists('../data/deepcraters.csv') else pd.read_csv('data/deepcraters.csv')
diviner = pd.read_csv('../data/diviner_global.csv') if os.path.exists('../data/diviner_global.csv') else pd.read_csv('data/diviner_global.csv')

# DeepCratersの年代インデックス（1〜5の数字）を、意味のわかる文字列にした列を追加しておく
AGE_NAMES = {
    1: '1:Pre-Nectarian(最も古い)',
    2: '2:Nectarian',
    3: '3:Imbrian',
    4: '4:Eratosthenian',
    5: '5:Copernican(最も新しい)',
}
deepcraters['Age_name'] = deepcraters['Age'].map(AGE_NAMES)

# データセットごとに「どの列を選べるか」「日本語での説明」を定義する
# ※ Robbins Crater DBには「深さ」の情報は含まれていません（要確認事項として実データを確認した結果、
#    緯度・経度・直径に関する列のみで、深さを表す列は存在しませんでした）。
datasets = {
    'クレーターの直径・形（Robbins Crater DB, 直径8km以上）': {
        'df': craters,
        'columns': {
            'lat': '緯度 [度]',
            'lon': '経度 [度]',
            'diam_km': '直径 [km]',
            'diam_major_km': '長径 [km]',
            'diam_minor_km': '短径 [km]',
            'eccentricity': '離心率（真円=0に近いほど丸い）',
            'ellipticity': '扁平率（真円=1に近いほど丸い）',
            'rim_arc_fraction': 'リムが検出できた割合（0〜1）',
        },
        'color_options': [],
    },
    'クレーターの推定年代（DeepCraters, 直径8km以上）': {
        'df': deepcraters,
        'columns': {
            'Lat': '緯度 [度]',
            'Lon': '経度 [度]',
            'Diam_km': '直径 [km]',
            'Age': '推定年代区分（1〜5、数字が大きいほど新しい）',
        },
        'color_options': ['Age_name', 'Flags_data'],
    },
    '月面の温度（Diviner, 全球0.5度グリッド）': {
        'df': diviner,
        'columns': {
            'lon': '経度 [度]',
            'lat': '緯度 [度]',
            'temp_noon_K': '正午の温度 [K]',
            'temp_midnight_K': '深夜0時の温度 [K]',
            'temp_diff_K': '昼夜の温度差 [K]',
        },
        'color_options': [],
    },
}

print('読み込み完了：')
for name, d in datasets.items():
    print(f' - {name}: {len(d["df"]):,} 件')

## 探索ツール

下のプルダウンでデータセットとX軸・Y軸を選ぶと、その場で散布図と基本統計量（平均・標準偏差・相関係数）が表示されます。
点の数が多いデータセットは、見やすさのため一部だけをランダムに抜き出して表示します（「表示点数の上限」で調整可）。

In [ ]:
dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    description='データセット:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
x_dropdown = widgets.Dropdown(description='X軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
y_dropdown = widgets.Dropdown(description='Y軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
color_dropdown = widgets.Dropdown(description='色分け:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
sample_slider = widgets.IntSlider(
    value=3000, min=500, max=20000, step=500,
    description='表示点数の上限:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
output = widgets.Output()


def update_variable_options(change=None):
    info = datasets[dataset_dropdown.value]
    options = [(label, col) for col, label in info['columns'].items()]
    x_dropdown.options = options
    y_dropdown.options = options
    x_dropdown.value = options[0][1]
    y_dropdown.value = options[1][1] if len(options) > 1 else options[0][1]
    color_dropdown.options = [('なし', None)] + [(c, c) for c in info['color_options']]
    color_dropdown.value = None
    draw_plot()


def draw_plot(change=None):
    with output:
        clear_output(wait=True)
        info = datasets[dataset_dropdown.value]
        df = info['df']
        x_col, y_col = x_dropdown.value, y_dropdown.value
        color_col = color_dropdown.value

        n = min(len(df), sample_slider.value)
        plot_df = df.sample(n=n, random_state=0) if len(df) > n else df

        fig, ax = plt.subplots(figsize=(7, 6))
        if color_col:
            categories = plot_df[color_col].astype('category')
            sc = ax.scatter(plot_df[x_col], plot_df[y_col], c=categories.cat.codes,
                             cmap='viridis', s=8, alpha=0.6)
            handles, _ = sc.legend_elements()
            ax.legend(handles, categories.cat.categories, title=color_col,
                       bbox_to_anchor=(1.05, 1), loc='upper left')
        else:
            ax.scatter(plot_df[x_col], plot_df[y_col], s=8, alpha=0.4)

        ax.set_xlabel(info['columns'].get(x_col, x_col))
        ax.set_ylabel(info['columns'].get(y_col, y_col))
        ax.set_title(f"{dataset_dropdown.value}\n(表示 {n:,} / 全 {len(df):,} 件)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        if x_col != y_col:
            corr = plot_df[[x_col, y_col]].corr().iloc[0, 1]
            print(f'相関係数 r = {corr:.3f}')
        stats = plot_df[[x_col, y_col]].describe().loc[['mean', 'std', 'min', 'max']]
        display(stats)


dataset_dropdown.observe(update_variable_options, names='value')
x_dropdown.observe(draw_plot, names='value')
y_dropdown.observe(draw_plot, names='value')
color_dropdown.observe(draw_plot, names='value')
sample_slider.observe(draw_plot, names='value')

update_variable_options()

display(widgets.VBox([
    dataset_dropdown,
    widgets.HBox([x_dropdown, y_dropdown, color_dropdown]),
    sample_slider,
    output,
]))

## 問いのヒント（迷ったときに）

正解ではなく、あくまで出発点の例です。自分で見つけた組み合わせを優先してください。

1. クレーターの直径と、離心率・扁平率（＝どれくらい丸いか）の関係
2. クレーターの緯度・経度分布のかたより（密集している場所とそうでない場所）
3. 緯度と正午の温度の関係
4. 同じ場所での「正午の温度」と「深夜0時の温度」の差（昼夜温度差）と、緯度との関係
5. クレーターの推定年代（Age）と、直径や分布との関係（DeepCratersのデータのみで完結）

> **注記**：当初の教材案には「クレーターの直径と深さの関係」という問いがありましたが、
> 実際にRobbins Crater Database (2018) のCSVを確認したところ、**深さ（Depth）を表す列は
> 含まれていません**（緯度・経度・直径・形状に関する列のみ）。そのため、この教材では
> 「直径と深さ」の代わりに「直径と離心率・扁平率」を候補にしています。

## データの出典

- Robbins, S. J. (2018). *A New Global Database of Lunar Impact Craters >1–2 km*. USGS Astrogeology Science Center.
  https://astrogeology.usgs.gov/search/map/Moon/Research/Craters/lunar_crater_database_robbins_2018
- Yang, C., Guan, R. (2020). *CE_DeepCraters* (Aged Lunar Crater Database). figshare.
  https://doi.org/10.6084/m9.figshare.12768539
- Williams, J.-P. et al. (2017). *The global surface temperatures of the Moon as measured by the
  Diviner Lunar Radiometer Experiment*. Icarus, 283, 300-325. データ配布：
  https://www.diviner.ucla.edu/data （UCLA Diviner Lunar Radiometer Experiment チーム提供、
  0.5 ppd全球ラスタープロダクト）